# Лабораторная работа 12. VAR и многомерные временные ряды

**Цель.** Исследовать многомерный временной ряд, проверить стационарность и построить VAR-прогноз.


## Ход работы

VAR отличается от одномерных моделей тем, что несколько рядов прогнозируются вместе. Поэтому перед моделью важно привести таблицу к числовому виду и проверить свойства каждого столбца.


## Подготовка

Подключаю инструменты для многомерного анализа и задаю возможные пути к CSV-файлу. Для VAR дальше нужны только числовые признаки без пропусков.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.api import VAR

np.random.seed(42)
csv_candidates = [Path('AirQualityUCI.csv'), Path('data/AirQualityUCI.csv')]
csv_path = next((path for path in csv_candidates if path.exists()), csv_candidates[0])


## Очистка и первичный график

Данные приводятся к числовому формату, пропуски интерполируются, затем несколько первых признаков выводятся на графики. Так можно быстро увидеть масштаб и динамику каждого ряда.


In [ ]:
if not csv_path.exists():
    print('CSV-файл не найден, VAR-анализ пропущен.')
else:
    df = pd.read_csv(csv_path, parse_dates=[['Date', 'Time']], sep=';', decimal=',')
    df['Date_Time'] = pd.to_datetime(df['Date_Time'], format='%d/%m/%Y %H.%M.%S', errors='coerce')
    data = df.drop(columns=['Date_Time']).apply(pd.to_numeric, errors='coerce')
    data.index = df['Date_Time']
    data = data.replace(-200, np.nan).interpolate().dropna(axis=1, how='all').dropna()
    numeric = data.select_dtypes(include='number').iloc[:, :4]
    print(numeric.head())
    numeric.plot(figsize=(12, 5), subplots=True, layout=(2, 2), legend=False)
    plt.tight_layout()
    plt.show()


## Стационарность и прогноз VAR

Для каждого признака считается ADF-тест, после чего VAR строит прогноз сразу по нескольким столбцам. Это позволяет учитывать взаимное влияние рядов, а не прогнозировать каждый отдельно.


In [ ]:
if csv_path.exists() and 'numeric' in globals() and len(numeric) > 30:
    for column in numeric.columns:
        pvalue = adfuller(numeric[column].dropna())[1]
        print(f'{column}: p-value ADF = {pvalue:.4f}')

    train = numeric.iloc[:-24]
    model = VAR(train)
    fitted = model.fit(maxlags=5, ic='aic')
    forecast = fitted.forecast(train.values[-fitted.k_ar:], steps=24)
    forecast = pd.DataFrame(forecast, columns=numeric.columns)
    print(forecast.head())


## Вывод

VAR подходит для ситуаций, где несколько временных рядов связаны между собой. Перед такой моделью особенно важны очистка данных и проверка стационарности, иначе связи между рядами будут оценены нестабильно.
